# Tomographic mapping notebook  
__Written by Haixing Fang, Jon Wright and James Ball__  
__Date: 21/02/2025__

This notebook will try to perform a point-by-point strain refinement from your tomographic-derived grain shapes.  

### NOTE: It is highly recommended to run this notebook on a Jupyter server with many cores and a lot of RAM.  
The compute_origins() function in particular runs locally and can be compute-intensive for large datasets.  
If this is a big scan (e.g 100 million+ 2D peaks), you should definitely refine on the cluster rather than locally.

Set `dset_path` to the saved DataSet from Anri's mapping notebook, including its run folder. As in S3DXRD, the path is loaded directly and the DataSet's saved grain and refinement paths are retained. Anri population hypotheses and the saved manual mask remain part of the refinement input.

In [ ]:
# this cell is tagged with parameters
from esrf_pathlib import ESRFPath as Path
import os, sys, importlib.metadata
# ImageD11 checkout selection follows the S3DXRD notebooks.
IMAGED11_PATH = '/home/esrf/charles27/Code/ImageD11'  # checkout folder within CHECKOUT_PATH
ANRI_PATH = '/home/esrf/charles27/Code/anri'
CHECKOUT_PATH = None  # None means guess the checkout parent folder
phase_str = 'Fe'
par_file = None  # None: use the DataSet calibration
anri_output_path = None  # None: <ds.analysispath>/anri
cif_file = '/data/id11/archive/file_cif/Fe.cif'
THREADS = 48
SEED_FRACTIONS = (0.993, 0.999, 0.9999)
ACTIVE_FRACTION = 0.999
LOCAL_POPULATION_MIN = 0.1

# Saved mapping DataSet, including the Anri run folder (loaded directly as in S3DXRD)
dset_path = '/data/visitor/ma7056/id11/20260624/PROCESSED_DATA/pureIron_2_heating_3/pureIron_2_heating_3_sff_after_20min0/anri_imaged11_s3dxrd_4/pureIron_2_heating_3_sff_after_20min0_dataset.h5'
analysispath = None  # retain ds.analysispath and all saved output paths

# manual override of y0 (ignores any value saved in dataset)
y0_manual = None
# or y0_manual = 0.0 (typical NSCOPE) or 13.5 (typical TDXRD)

# default options for the single-valued map (shouldn't need to modify this)
default_npks = 20
default_nuniq = 20

# refinement tolerances
hkl_tol_origins = 0.05
# ^ we forgot to square this before!
hkl_tol_refine = 0.1
hkl_tol_refine_merged = 0.05
ds_tol = 0.004
ifrac = 0.0003
rings_to_refine = list(range(11))
drop_input_peaks = False

# ^ for large datasets, delete cf_2d from memory after choosing peaks to refine with
use_origin_found_only=True
# use cluster for refinement or run locally?
use_cluster = False

In [ ]:
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[key]='1'
os.environ['NUMBA_NUM_THREADS']=str(THREADS)
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE','false')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION','0.30')
if hasattr(os, 'sched_getaffinity'):
    os.sched_setaffinity(0, set(sorted(os.sched_getaffinity(0))[:THREADS]))
if IMAGED11_PATH is not None:
    exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())
    PYTHONPATH = setup_ImageD11_from_git(CHECKOUT_PATH, IMAGED11_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    # Prefer the local checkout when running from it, as in S3DXRD.
    from pathlib import Path as LocalPath
    local_candidates = [LocalPath.cwd(), LocalPath.cwd() / 'ImageD11'] + list(LocalPath.cwd().parents)
    for candidate in local_candidates:
        if (candidate / 'ImageD11' / 'nbGui' / 'anri' / 'tomo_3_refinement.ipynb').is_file():
            loaded = sys.modules.get('ImageD11')
            if loaded is not None and (getattr(loaded, '__file__', None) is None or LocalPath(loaded.__file__).resolve().parent != candidate / 'ImageD11'):
                raise RuntimeError('Restart this notebook kernel to use the modified ImageD11 checkout')
            sys.path.insert(0, str(candidate))
            PYTHONPATH = str(candidate)
            break
print(PYTHONPATH)
if ANRI_PATH is not None:
    sys.path.insert(0, str(ANRI_PATH))
import ImageD11
import h5py, numpy as np
import ImageD11.sinograms.dataset as dataset
import numba
numba.set_num_threads(THREADS)
def log(message):
    print(message,flush=True)
def snapshot(fig,name):
    from IPython.display import display
    import matplotlib.pyplot as plt
    fig.savefig(RUN/(name+'.png'),dpi=120,bbox_inches='tight'); display(fig); plt.close(fig)


In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt

import ImageD11.sinograms.dataset
from ImageD11.sinograms.sinogram import read_h5
from ImageD11.sinograms.tensor_map import TensorMap
from ImageD11.sinograms.pbp_refine import PBPRefine

import ImageD11.nbGui.nb_utils as utils

%matplotlib ipympl

# Load data
## Dataset

In [ ]:
ds = dataset.load(dset_path)
SOURCE_DATASET = Path(dset_path).absolute()
SOURCE_SPARSE = Path(ds.sparsefile)
SOURCE_GRAINS = Path(ds.grainsfile)
PARFILE = Path(par_file if par_file is not None else ds.parfile)
if par_file is not None:
    ds.parfile = str(PARFILE)
NAME, SAMPLE, SCAN = ds.dsname, ds.sample, ds.dset
PHASE = phase_str
DATA = SOURCE_DATASET.parent
RUN = Path(analysispath) if analysispath is not None else Path(ds.analysispath)
ANRI = Path(anri_output_path) if anri_output_path is not None else RUN/'anri'
VALIDATION = RUN/'validation'
VARIANT_DATASET = Path(ds.dsfile) if analysispath is None else RUN/(NAME+'_dataset.h5')
ANRI_FIT = ANRI/(NAME+'_fit.h5')
ANRI_POP = ANRI/(NAME+'_populations.h5')
CIF = Path(cif_file)
print(ds)


In [ ]:
ds_check=ds
assert not Path(os.path.splitext(ds_check.refoutfile)[0] + '_{}.h5'.format(phase_str)).exists(), 'Refinement exists; use a new run'

In [ ]:
print(ds)

## Phases
If the parameter file was a json, we can access the unit cells via `ds.phases.unitcells`

In [ ]:
ds.phases = ds.get_phases_from_disk()
ds.phases.unitcells

In [ ]:
ucell = ds.phases.unitcells[phase_str]
print(ucell)

## Peaks

In [ ]:
cf_2d = ds.get_cf_2d()
ds.update_colfile_pars(cf_2d)  # computes geometry, needed for filtration
print(f"Read {cf_2d.nrows} 2D peaks")

## Grains

In [ ]:
grainsinos = read_h5(ds.grainsfile, ds, phase_str)
grains = [gs.grain for gs in grainsinos]

## TensorMap

In [ ]:
tensor_map = TensorMap.from_h5(ds.grainsfile, h5group='TensorMap_' + phase_str)
tensor_map.plot('phase_ids')

In [ ]:
# make a PBPMap from our TensorMap
# fills voxels that have grains with npks = 20 and nuniq = 20
pmap = tensor_map.to_pbpmap(z_layer=0, default_npks=default_npks, default_nuniq=default_nuniq)
# Add local Anri orientation hypotheses, not accepted counts or copied reference orientations.
from esrf_pathlib import ESRFPath as Path
from ImageD11.sinograms.point_by_point import PBPMap
from ImageD11.sinograms.geometry import recon_to_step
anri_folder = ANRI
anri_name = NAME
with h5py.File(anri_folder / (anri_name + '_populations.h5'), 'r') as h:
    local_u = h['U_pop'][()]
    local_frac = h['frac'][()]
with h5py.File(anri_folder / (anri_name + '_fit.h5'), 'r') as h:
    local_total = h['f'][()].sum(1)
with h5py.File(ds.grainsfile, 'r') as h:
    user_mask = h['masks/' + phase_str][()].astype(bool)
orthogonal = np.max(np.abs(np.matmul(local_u.swapaxes(-1,-2),local_u)-np.eye(3)),axis=(-2,-1))<1e-4
rotation_ok = np.isfinite(local_u).all(axis=(-2,-1)) & orthogonal & (np.abs(np.linalg.det(local_u)-1.0)<1e-4)
offer = rotation_ok & (local_frac>=LOCAL_POPULATION_MIN) & (local_total[:,None]>0) & user_mask.ravel()[:,None]
voxel, population = np.nonzero(offer)
initial_ubi = np.linalg.inv(np.matmul(local_u[voxel,population],ucell.B))
si,sj = recon_to_step(voxel // user_mask.shape[1],voxel % user_mask.shape[1],user_mask.shape)
extra = dict(i=si,j=sj,ntotal=np.full(len(voxel),default_npks),nuniq=np.full(len(voxel),default_nuniq))
for a in range(3):
    for b in range(3): extra['ubi'+str(a)+str(b)] = initial_ubi[:,a,b]
augmented = PBPMap(new=True)
augmented.nrows = pmap.nrows + len(voxel)
for title in pmap.titles: augmented.addcolumn(np.concatenate((getattr(pmap,title),extra[title])),title)
pmap = augmented
print('Added Anri local hypotheses:',len(voxel),'total multivalued input rows:',pmap.nrows)
print('Initial ntotal/nuniq values are placeholders, not measurements or acceptance')

In [ ]:
pmap.choose_best(1)
pmap.plot_best(1)

In [ ]:
# Priority order (decreasing)
# manual override in first cell
# ds attribute

# fails if no y0 provided at all

if y0_manual is not None:
    print('Using manually supplied y0')
    y0 = y0_manual
else:
    if hasattr(ds, 'y0') and ds.y0 is not None:
        print('Using ds.y0')
        y0 = ds.y0
    else:
        raise ValueError('You must manually supply a y0 value in the first cell to continue.')
print('Using y0:', y0)
# if we got here, save to ds.y0 - future refinements should use the same one
ds.y0 = y0
ds.save()

# Refinement

In [ ]:
# set up a refinement manager object

refine = PBPRefine(dset=ds,
                   y0=y0, hkl_tol_origins=hkl_tol_origins, hkl_tol_refine=hkl_tol_refine, hkl_tol_refine_merged=hkl_tol_refine_merged,
                   ds_tol=ds_tol, ifrac=ifrac, phase_name=phase_str, forref=rings_to_refine,
                   beam_size=None)  # use step size as beam size - important for regularisation
# change the default paths of the refinement manager to append the phase name
refine.own_filename = os.path.splitext(refine.own_filename)[0] + f'_{phase_str}.h5'
refine.icolf_filename = os.path.splitext(refine.icolf_filename)[0] + f'_{phase_str}.h5'
refine.pbpmap_filename = os.path.splitext(refine.pbpmap_filename)[0] + f'_{phase_str}.h5'
refine.refinedmap_filename = os.path.splitext(refine.refinedmap_filename)[0] + f'_{phase_str}.h5'

In [ ]:
# tell it which point-by-point map we are refining
refine.setmap(pmap)

# or load from disk:
# refine.loadmap()

In [ ]:
# set the mask from minimum peak values
# anything greater than 0 should be accepted
with h5py.File(ds.grainsfile, 'r') as h:
    refine.mask = h['masks/' + phase_str][()].astype(bool)
assert refine.mask.shape == refine.sx_grid.shape
print('Exact manual refinement support:',int(refine.mask.sum()))

In [ ]:
# generate a single-valued map to refine on
refine.setsingle(refine.pbpmap, minpeaks=1)

In [ ]:
# choose 2D peaks to refine with
refine.setpeaks(cf_2d, keep_colf=not drop_input_peaks)

# or load from disk:
# refine.loadpeaks()

In [ ]:
if drop_input_peaks:
    del cf_2d
    import gc; gc.collect()
    print("released input peaks")

## Setting up peaks

In [ ]:
# plot the peaks you selected
refine.iplot()

## Compute peak diffraction origins

In [ ]:
# compute diffraction origins - these will be added as a column to refine.icolf
# will then save the new column to disk to avoid re-computation
refine.get_origins(nthreads=4, nchunks=4)

In [ ]:
# after refine.get_origins()
if use_origin_found_only:
    n0 = refine.icolf.nrows
    refine.icolf = refine.icolf.copyrows(refine.icolf.origin_found == 1)
    print(f"origin_found filter: kept {refine.icolf.nrows} / {n0} peaks")

## Run refinement

In [ ]:
# run the refinement
# if compute_origins took more than a couple of minutes to run, I suggest setting use_cluster=True below
# otherwise if you asked for lots of cores and RAM on this Jupyter instance, you can run it locally (use_cluster=False)
refine.run_refine(use_cluster=use_cluster, pythonpath=PYTHONPATH, nthreads=4, nchunks=4)

# Export

In [ ]:
# make sure refinement is saved to disk

if not use_cluster:
    refine.to_h5(save_peaks=True)

ds.save()